In [113]:
# Imports

import os
import sys
import rdkit
import joblib
import logging
import warnings
import subprocess
import numpy as np
import pandas as pd
import deepchem as dc
from time import sleep
from scipy import stats
import pubchempy as pcp
from pathlib import Path
from tqdm.auto import tqdm
from typing import Optional
from __future__ import division
import matplotlib.pyplot as plt
from collections import Counter
from xgboost import XGBClassifier
from IPython.display import display
from pandarallel import pandarallel
from __future__ import absolute_import
from __future__ import unicode_literals
from rdkit.Chem.Draw import IPythonConsole
from deepchem.splits import ButinaSplitter
from dimorphite_dl import protonate_smiles
from rdkit.Chem.EState import Fingerprinter
from mordred import Calculator, descriptors
from rdkit.DataStructs import BitVectToText
from rdkit.Chem.MACCSkeys import GenMACCSKeys
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from rdkit.ML.Descriptors import MoleculeDescriptors 
from rdkit import Chem, DataStructs, rdBase, RDLogger
from rdkit.Chem.MolStandardize import rdMolStandardize
from sklearn.feature_selection import VarianceThreshold
from sklearn.experimental import enable_halving_search_cv
from rdkit.Chem import AllChem, Draw, Descriptors, PandasTools, AddHs, inchi, MolStandardize
from sklearn.model_selection import HalvingRandomSearchCV, cross_val_predict, StratifiedKFold, RandomizedSearchCV, train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, balanced_accuracy_score, f1_score, matthews_corrcoef, precision_score, average_precision_score, roc_auc_score, ConfusionMatrixDisplay, roc_curve, recall_score

# Suppress warnings

warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore")
RDLogger.DisableLog('rdApp.*')

In [ ]:
# Define data

root_dir = '/Users/HQ/Repositories/DILI/'
data_dir = os.path.join(root_dir, 'Data')


binary_mapping = {
    'Most': 1,
    'Less': 1,
    'No': 0
}


features = pd.read_csv(f'{data_dir}/DILI_rank_2_features.csv')
features = features.drop(columns=['Unnamed: 0', 'CompoundName'])
features['Label'] = features['Label'].replace(binary_mapping)

Goldstandard_features = pd.read_csv(f'/Users/HQ/Repositories/PredictorDILI/DILI/Test_data_DILIst.csv')
Goldstandard_features = Goldstandard_features.drop(columns=['Source_rank', 'Source', 'Data', 'MACCS0'])
Goldstandard_features = Goldstandard_features.rename(columns={'TOXICITY': 'Label'})

combination = pd.concat([Goldstandard_features, features], join='inner', ignore_index=True)
combination = combination.drop_duplicates(subset=['InChIKey14'], keep='first')

features = combination.copy()

In [23]:
# Define models

MODEL_PATH = Path("/Users/HQ/Repositories/PredictorDILI/Local_Implementation_v4/models")   # ← change if you put it elsewhere

# Load all 11 proxy models
model_files = {
    "bestlivmodel_3_model.sav":  "human_hepatotoxicity",
    "bestlivmodel_5_model.sav":  "animal_hepatotoxicity_A",
    "bestlivmodel_6_model.sav":  "animal_hepatotoxicity_B",
    "bestlivmodel_7_model.sav":  "preclinical_hepatotoxicity",
    "bestlivmodel_8_model.sav":  "diverse_DILI_A",
    "bestlivmodel_10_model.sav": "diverse_DILI_C",
    "bestlivmodel_11_model.sav": "BSEP_inhibition",
    "bestlivmodel_14_model.sav": "mitochondrial_toxicity",
    "bestlivmodel_15_model.sav": "reactive_metabolite",
    
    # The two regression models
    "bestlivmodel_median pMolar total plasma concentration_model.sav":   "Cmax_total",
    "bestlivmodel_median pMolar unbound plasma concentration_model.sav": "Cmax_unbound",
}

# Load all models
proxy_models = {}
for filename, name in model_files.items():
    path = MODEL_PATH / filename
    if path.exists():
        proxy_models[name] = joblib.load(path)

for name, model in proxy_models.items():
    try:
        n = model.n_features_in_
        print(f"{name:30} → {n} features")
    except:
        print(f"{name:30} → UNKNOWN")

# # print(f"\nSuccessfully loaded {len(proxy_models)} proxy models")

human_hepatotoxicity           → 857 features
animal_hepatotoxicity_A        → 857 features
animal_hepatotoxicity_B        → 857 features
preclinical_hepatotoxicity     → 857 features
diverse_DILI_A                 → 857 features
diverse_DILI_C                 → 856 features
BSEP_inhibition                → 857 features
mitochondrial_toxicity         → 857 features
reactive_metabolite            → 857 features
Cmax_total                     → 857 features
Cmax_unbound                   → 857 features


In [41]:
# Get which feature columns are reqeusted by each model

expected_columns = ['ABC', 'ABCGG', 'nAcid', 'nBase', 'SpAbs_A', 'SpDiam_A', 'SpAD_A', 'LogEE_A', 'VE1_A', 'VE3_A', 'VR1_A', 'VR2_A', 'nAromAtom', 'nAromBond', 'nAtom', 'nHeavyAtom', 'nBridgehead', 'nHetero', 'nH', 'nC', 'nN', 'nO', 'nS', 'nF', 'nCl', 'nX', 'ATS0dv', 'ATS1dv', 'ATS2dv', 'ATS3dv', 'ATS4dv', 'ATS5dv', 'ATS6dv', 'ATS7dv', 'ATS8dv', 'ATS0d', 'ATS1d', 'ATS2d', 'ATS3d', 'ATS4d', 'ATS5d', 'ATS6d', 'ATS7d', 'ATS8d', 'ATS0Z', 'ATS1Z', 'ATS2Z', 'ATS3Z', 'ATS4Z', 'ATS5Z', 'ATS6Z', 'ATS7Z', 'ATS8Z', 'ATS0m', 'ATS1m', 'ATS2m', 'ATS3m', 'ATS4m', 'ATS5m', 'ATS6m', 'ATS7m', 'ATS8m', 'ATS0v', 'ATS1v', 'ATS2v', 'ATS3v', 'ATS4v', 'ATS5v', 'ATS6v', 'ATS7v', 'ATS8v', 'ATS0se', 'ATS1se', 'ATS2se', 'ATS3se', 'ATS4se', 'ATS5se', 'ATS6se', 'ATS7se', 'ATS8se', 'ATS0pe', 'ATS1pe', 'ATS2pe', 'ATS3pe', 'ATS4pe', 'ATS5pe', 'ATS6pe', 'ATS7pe', 'ATS8pe', 'ATS0are', 'ATS1are', 'ATS2are', 'ATS3are', 'ATS4are', 'ATS5are', 'ATS6are', 'ATS7are', 'ATS8are', 'ATS0p', 'ATS1p', 'ATS2p', 'ATS3p', 'ATS4p', 'ATS5p', 'ATS6p', 'ATS7p', 'ATS8p', 'ATS0i', 'ATS1i', 'ATS2i', 'ATS3i', 'ATS4i', 'ATS5i', 'ATS6i', 'ATS7i', 'ATS8i', 'AATS0dv', 'AATS1dv', 'AATS0d', 'AATS1d', 'AATS0Z', 'AATS1Z', 'AATS0m', 'AATS1m', 'AATS0v', 'AATS1v', 'AATS0se', 'AATS1se', 'AATS0pe', 'AATS1pe', 'AATS0are', 'AATS1are', 'AATS0p', 'AATS0i', 'AATS1i', 'ATSC0dv', 'ATSC1dv', 'ATSC2dv', 'ATSC3dv', 'ATSC4dv', 'ATSC5dv', 'ATSC6dv', 'ATSC7dv', 'ATSC8dv', 'ATSC0d', 'ATSC1d', 'ATSC2d', 'ATSC3d', 'ATSC4d', 'ATSC5d', 'ATSC6d', 'ATSC7d', 'ATSC8d', 'ATSC0Z', 'ATSC1Z', 'ATSC2Z', 'ATSC3Z', 'ATSC4Z', 'ATSC5Z', 'ATSC6Z', 'ATSC7Z', 'ATSC8Z', 'ATSC0m', 'ATSC1m', 'ATSC2m', 'ATSC3m', 'ATSC4m', 'ATSC5m', 'ATSC6m', 'ATSC7m', 'ATSC8m', 'ATSC0v', 'ATSC1v', 'ATSC2v', 'ATSC3v', 'ATSC4v', 'ATSC5v', 'ATSC6v', 'ATSC7v', 'ATSC8v', 'ATSC0se', 'ATSC1se', 'ATSC2se', 'ATSC3se', 'ATSC4se', 'ATSC5se', 'ATSC6se', 'ATSC7se', 'ATSC8se', 'ATSC0pe', 'ATSC1pe', 'ATSC2pe', 'ATSC3pe', 'ATSC4pe', 'ATSC5pe', 'ATSC6pe', 'ATSC7pe', 'ATSC8pe', 'ATSC0are', 'ATSC1are', 'ATSC2are', 'ATSC3are', 'ATSC4are', 'ATSC5are', 'ATSC6are', 'ATSC7are', 'ATSC8are', 'ATSC0p', 'ATSC1p', 'ATSC2p', 'ATSC3p', 'ATSC4p', 'ATSC5p', 'ATSC6p', 'ATSC7p', 'ATSC8p', 'ATSC0i', 'ATSC1i', 'ATSC2i', 'ATSC3i', 'ATSC4i', 'ATSC5i', 'ATSC6i', 'ATSC7i', 'ATSC8i', 'AATSC0dv', 'AATSC1dv', 'AATSC0Z', 'AATSC1Z', 'AATSC0m', 'AATSC1m', 'AATSC0v', 'AATSC1v', 'AATSC0i', 'BCUTdv-1h', 'BCUTdv-1l', 'BCUTd-1h', 'BCUTZ-1h', 'BCUTm-1h', 'BCUTv-1h', 'BCUTv-1l', 'BCUTp-1h', 'BCUTi-1h', 'BalabanJ', 'SpAbs_DzZ', 'SpMax_DzZ', 'SpDiam_DzZ', 'SpAD_DzZ', 'SpMAD_DzZ', 'LogEE_DzZ', 'SM1_DzZ', 'VE1_DzZ', 'VE3_DzZ', 'VR1_DzZ', 'VR2_DzZ', 'SpAbs_Dzm', 'SpMax_Dzm', 'SpDiam_Dzm', 'SpAD_Dzm', 'SpMAD_Dzm', 'LogEE_Dzm', 'SM1_Dzm', 'VE1_Dzm', 'VE3_Dzm', 'VR1_Dzm', 'VR2_Dzm', 'SpAbs_Dzv', 'SpMax_Dzv', 'SpDiam_Dzv', 'SpAD_Dzv', 'SpMAD_Dzv', 'LogEE_Dzv', 'SM1_Dzv', 'VE1_Dzv', 'VE3_Dzv', 'VR1_Dzv', 'VR2_Dzv', 'SpAbs_Dzse', 'SpMax_Dzse', 'SpDiam_Dzse', 'SpAD_Dzse', 'SpMAD_Dzse', 'LogEE_Dzse', 'SM1_Dzse', 'VE1_Dzse', 'VE3_Dzse', 'VR1_Dzse', 'VR2_Dzse', 'SpAbs_Dzpe', 'SpMax_Dzpe', 'SpDiam_Dzpe', 'SpAD_Dzpe', 'SpMAD_Dzpe', 'LogEE_Dzpe', 'SM1_Dzpe', 'VE1_Dzpe', 'VE3_Dzpe', 'VR1_Dzpe', 'VR2_Dzpe', 'SpAbs_Dzare', 'SpMax_Dzare', 'SpDiam_Dzare', 'SpAD_Dzare', 'SpMAD_Dzare', 'LogEE_Dzare', 'SM1_Dzare', 'VE1_Dzare', 'VE3_Dzare', 'VR1_Dzare', 'VR2_Dzare', 'SpAbs_Dzp', 'SpMax_Dzp', 'SpDiam_Dzp', 'SpAD_Dzp', 'SpMAD_Dzp', 'LogEE_Dzp', 'SM1_Dzp', 'VE1_Dzp', 'VE3_Dzp', 'VR1_Dzp', 'VR2_Dzp', 'SpAbs_Dzi', 'SpMax_Dzi', 'SpDiam_Dzi', 'SpAD_Dzi', 'SpMAD_Dzi', 'LogEE_Dzi', 'SM1_Dzi', 'VE1_Dzi', 'VE3_Dzi', 'VR1_Dzi', 'VR2_Dzi', 'BertzCT', 'nBonds', 'nBondsO', 'nBondsS', 'nBondsD', 'nBondsA', 'nBondsM', 'nBondsKS', 'nBondsKD', 'C1SP2', 'C2SP2', 'C3SP2', 'C1SP3', 'C2SP3', 'C3SP3', 'C4SP3', 'Xch-7d', 'Xch-7dv', 'Xc-3d', 'Xc-5d', 'Xc-6d', 'Xc-3dv', 'Xc-5dv', 'Xpc-4d', 'Xpc-5d', 'Xpc-6d', 'Xpc-4dv', 'Xpc-5dv', 'Xpc-6dv', 'Xp-1d', 'Xp-2d', 'Xp-3d', 'Xp-4d', 'Xp-5d', 'Xp-6d', 'Xp-7d', 'Xp-1dv', 'Xp-2dv', 'Xp-3dv', 'Xp-4dv', 'Xp-5dv', 'Xp-6dv', 'Xp-7dv', 'SZ', 'Sm', 'Sv', 'Sse', 'Spe', 'Sare', 'Sp', 'Si', 'SpAbs_D', 'SpMax_D', 'SpDiam_D', 'SpAD_D', 'SpMAD_D', 'LogEE_D', 'VE1_D', 'VE3_D', 'VR1_D', 'VR2_D', 'NsCH3', 'NssCH2', 'NdsCH', 'NaaCH', 'NsssCH', 'NdssC', 'NaasC', 'NaaaC', 'NssssC', 'NsNH2', 'NssNH', 'NaaNH', 'NsssNH', 'NdsN', 'NaaN', 'NaasN', 'NsOH', 'NdO', 'NssO', 'NsF', 'NssS', 'NsCl', 'SsCH3', 'SdCH2', 'SssCH2', 'StCH', 'SdsCH', 'SaaCH', 'SsssCH', 'StsC', 'SdssC', 'SaasC', 'SaaaC', 'SssssC', 'SsNH3', 'SsNH2', 'SssNH2', 'SdNH', 'SssNH', 'SaaNH', 'StN', 'SsssNH', 'SdsN', 'SaaN', 'SsssN', 'SaasN', 'SsOH', 'SdO', 'SssO', 'SaaO', 'SsF', 'SdsssP', 'SsSH', 'SdS', 'SssS', 'SddssS', 'SsCl', 'SsBr', 'SsI', 'ECIndex', 'ETA_alpha', 'ETA_beta', 'ETA_beta_s', 'ETA_beta_ns', 'ETA_beta_ns_d', 'ETA_eta', 'AETA_eta', 'ETA_eta_L', 'ETA_eta_R', 'AETA_eta_R', 'ETA_eta_RL', 'ETA_eta_F', 'ETA_eta_FL', 'ETA_dBeta', 'fragCpx', 'nHBAcc', 'nHBDon', 'IC1', 'IC2', 'IC3', 'IC4', 'IC5', 'TIC0', 'TIC1', 'TIC2', 'TIC3', 'TIC4', 'TIC5', 'CIC0', 'CIC1', 'CIC2', 'CIC3', 'CIC4', 'CIC5', 'MIC0', 'MIC1', 'MIC2', 'MIC3', 'MIC4', 'MIC5', 'ZMIC0', 'ZMIC1', 'ZMIC2', 'ZMIC3', 'ZMIC4', 'ZMIC5', 'Lipinski', 'GhoseFilter', 'FilterItLogS', 'VMcGowan', 'LabuteASA', 'PEOE_VSA1', 'PEOE_VSA2', 'PEOE_VSA3', 'PEOE_VSA4', 'PEOE_VSA5', 'PEOE_VSA6', 'PEOE_VSA7', 'PEOE_VSA8', 'PEOE_VSA9', 'PEOE_VSA10', 'PEOE_VSA11', 'PEOE_VSA12', 'PEOE_VSA13', 'SMR_VSA1', 'SMR_VSA2', 'SMR_VSA3', 'SMR_VSA4', 'SMR_VSA5', 'SMR_VSA6', 'SMR_VSA7', 'SMR_VSA9', 'SlogP_VSA1', 'SlogP_VSA2', 'SlogP_VSA3', 'SlogP_VSA4', 'SlogP_VSA5', 'SlogP_VSA6', 'SlogP_VSA7', 'SlogP_VSA8', 'SlogP_VSA10', 'SlogP_VSA11', 'EState_VSA1', 'EState_VSA2', 'EState_VSA3', 'EState_VSA4', 'EState_VSA5', 'EState_VSA6', 'EState_VSA7', 'EState_VSA8', 'EState_VSA9', 'EState_VSA10', 'VSA_EState1', 'VSA_EState2', 'VSA_EState3', 'VSA_EState4', 'VSA_EState5', 'VSA_EState6', 'VSA_EState7', 'VSA_EState8', 'VSA_EState9', 'MID', 'MID_h', 'MID_C', 'MID_N', 'MID_O', 'MID_X', 'MPC2', 'MPC3', 'MPC4', 'MPC5', 'MPC6', 'MPC7', 'MPC8', 'MPC9', 'MPC10', 'TMPC10', 'piPC1', 'piPC2', 'piPC3', 'piPC4', 'piPC5', 'piPC6', 'piPC7', 'piPC8', 'piPC9', 'piPC10', 'TpiPC10', 'apol', 'bpol', 'nRing', 'n5Ring', 'n6Ring', 'nG12Ring', 'nHRing', 'n5HRing', 'n6HRing', 'nG12HRing', 'naRing', 'n5aRing', 'n6aRing', 'naHRing', 'n5aHRing', 'n6aHRing', 'nARing', 'n5ARing', 'n6ARing', 'nG12ARing', 'nAHRing', 'n5AHRing', 'n6AHRing', 'nG12AHRing', 'nFRing', 'nG12FRing', 'nFHRing', 'nFARing', 'nG12FARing', 'nFAHRing', 'nRot', 'SLogP', 'SMR', 'TopoPSA(NO)', 'TopoPSA', 'GGI1', 'GGI2', 'GGI3', 'GGI4', 'GGI5', 'GGI6', 'GGI7', 'Diameter', 'Radius', 'MWC01', 'MWC02', 'MWC03', 'MWC04', 'MWC05', 'MWC06', 'MWC07', 'MWC08', 'MWC09', 'MWC10', 'TMWC10', 'SRW02', 'SRW03', 'SRW04', 'SRW05', 'SRW06', 'SRW07', 'SRW08', 'SRW09', 'SRW10', 'TSRW10', 'MW', 'AMW', 'WPath', 'WPol', 'Zagreb1', 'Zagreb2', 'mZagreb2', 'Mfp1', 'Mfp13', 'Mfp80', 'Mfp114', 'Mfp147', 'Mfp216', 'Mfp222', 'Mfp227', 'Mfp231', 'Mfp249', 'Mfp283', 'Mfp294', 'Mfp310', 'Mfp314', 'Mfp322', 'Mfp350', 'Mfp378', 'Mfp389', 'Mfp392', 'Mfp397', 'Mfp486', 'Mfp519', 'Mfp561', 'Mfp591', 'Mfp650', 'Mfp656', 'Mfp675', 'Mfp694', 'Mfp695', 'Mfp715', 'Mfp718', 'Mfp725', 'Mfp739', 'Mfp745', 'Mfp794', 'Mfp807', 'Mfp816', 'Mfp841', 'Mfp875', 'Mfp926', 'Mfp935', 'Mfp1017', 'Mfp1019', 'Mfp1028', 'Mfp1039', 'Mfp1057', 'Mfp1060', 'Mfp1088', 'Mfp1114', 'Mfp1143', 'Mfp1152', 'Mfp1162', 'Mfp1171', 'Mfp1199', 'Mfp1226', 'Mfp1257', 'Mfp1274', 'Mfp1292', 'Mfp1309', 'Mfp1325', 'Mfp1357', 'Mfp1366', 'Mfp1380', 'Mfp1385', 'Mfp1386', 'Mfp1391', 'Mfp1444', 'Mfp1452', 'Mfp1476', 'Mfp1536', 'Mfp1564', 'Mfp1602', 'Mfp1607', 'Mfp1683', 'Mfp1690', 'Mfp1722', 'Mfp1729', 'Mfp1738', 'Mfp1750', 'Mfp1754', 'Mfp1855', 'Mfp1873', 'Mfp1911', 'Mfp1917', 'Mfp1921', 'Mfp1928', 'Mfp1970', 'Mfp1991', 'Mfp2031', 'MACCS49', 'MACCS50', 'MACCS53', 'MACCS54', 'MACCS57', 'MACCS62', 'MACCS65', 'MACCS66', 'MACCS72', 'MACCS74', 'MACCS75', 'MACCS76', 'MACCS77', 'MACCS79', 'MACCS80', 'MACCS81', 'MACCS82', 'MACCS83', 'MACCS84', 'MACCS85', 'MACCS86', 'MACCS87', 'MACCS88', 'MACCS89', 'MACCS90', 'MACCS91', 'MACCS92', 'MACCS93', 'MACCS94', 'MACCS95', 'MACCS96', 'MACCS97', 'MACCS98', 'MACCS99', 'MACCS100', 'MACCS101', 'MACCS102', 'MACCS103', 'MACCS104', 'MACCS105', 'MACCS106', 'MACCS107', 'MACCS108', 'MACCS109', 'MACCS110', 'MACCS111', 'MACCS112', 'MACCS113', 'MACCS114', 'MACCS115', 'MACCS116', 'MACCS117', 'MACCS118', 'MACCS119', 'MACCS120', 'MACCS121', 'MACCS122', 'MACCS123', 'MACCS124', 'MACCS125', 'MACCS126', 'MACCS127', 'MACCS128', 'MACCS129', 'MACCS130', 'MACCS131', 'MACCS132', 'MACCS133', 'MACCS134', 'MACCS135', 'MACCS136', 'MACCS137', 'MACCS138', 'MACCS139', 'MACCS140', 'MACCS141', 'MACCS142', 'MACCS143', 'MACCS144', 'MACCS145', 'MACCS146', 'MACCS147', 'MACCS148', 'MACCS149', 'MACCS150', 'MACCS151', 'MACCS152', 'MACCS153', 'MACCS154', 'MACCS155', 'MACCS156', 'MACCS157', 'MACCS158', 'MACCS159', 'MACCS160', 'MACCS161', 'MACCS162', 'MACCS163', 'MACCS164', 'MACCS165', 'PSA', 'n_rot_bonds', 'n_rings', 'n_ar_rings', 'n_HBA', 'n_HBD', 'Fsp3', 'logP', 'NHOHCount', 'NOCount', 'NumHeteroatoms', 'n_positive', '_n_negative', 'n_ring_asmbl', 'n_stereo']

X_filtered = features[expected_columns].copy()

lowest_var_column = X_filtered.var().idxmin()


In [42]:
# Generate_proxy_features

def generate_proxy_features(X_matrix):
    results = {}
    
    for name, model in tqdm(proxy_models.items(), desc="Generating proxies"):
        if name == "diverse_DILI_C":                    # ← the special model
            X_use = X_matrix.drop(columns=[lowest_var_column])
            results[name] = model.predict_proba(X_use)[:, 1]
        elif "Cmax" in name:
            results[name] = model.predict(X_matrix)
        else:
            results[name] = model.predict_proba(X_matrix)[:, 1]
    
    return pd.DataFrame(results)

proxy_df = generate_proxy_features(X_filtered)

proxy_df.columns = [
    'proxy_human', 'proxy_animal_A', 'proxy_animal_B', 'proxy_preclinical',
    'proxy_diverse_A', 'proxy_diverse_C', 'proxy_BSEP', 'proxy_mitotox',
    'proxy_reactive', 'proxy_Cmax_total', 'proxy_Cmax_unbound'
]



Generating proxies:   0%|          | 0/11 [00:00<?, ?it/s]

In [ ]:
# Remove invalid entries

bad_rows = []
for idx, smiles in enumerate(features["protonated_smiles_r"]):
    if pd.isna(smiles) or str(smiles).strip() in ["", "None", "INVALID", "nan"]:
        bad_rows.append((idx, smiles))
        continue
    
    mol = Chem.MolFromSmiles(str(smiles))
    if mol is None:
        bad_rows.append((idx, smiles))

print(f"Found {len(bad_rows)} invalid / failed SMILES:")
for idx, s in bad_rows:
    print(f"Row {idx}: {s}")

if bad_rows:
    bad_indices = [idx for idx, _ in bad_rows]
    features = features.drop(index=bad_indices).reset_index(drop=True)
    print(f"\nDropped {len(bad_indices)} bad rows. New size: {len(features)}")

Found 15 invalid / failed SMILES:
Row 4: CC1OC1[P](=O)(=O)O
Row 27: O=[P](=O)(O)C(Cl)(Cl)[P](=O)(=O)O
Row 48: CC(C)c1cccc(C(C)C)c1OCO[P](=O)(=O)O
Row 66: CC(Cn1cnc2c(=N)[nH]cnc21)OC[P](=O)(=O)O
Row 109: N=c1[nH]cnc2c1ncn2CCOC[P](=O)(=O)O
Row 461: [NH3+]CCCC(O)([P](=O)(=O)O)[P](=O)(=O)O
Row 583: O=[P](=O)(O)OC(Cn1cncn1)(Cn1cncn1)c1ccc(F)cc1F
Row 702: O=C1NC(c2ccccc2)(c2ccccc2)C(=O)[NH+]1CO[P](=O)(=O)O
Row 762: CC(O)([P](=O)(=O)O)[P](=O)(=O)O
Row 848: [NH3+]CCC(O)([P](=O)(=O)O)[P](=O)(=O)O
Row 859: O=C([O-])[P](=O)(=O)O
Row 919: Cc1ncc(CO[P](=O)(=O)O)c(C=O)c1O
Row 1026: N=c1ccn(CC(CO)OC[P](=O)(=O)O)c(=O)[nH]1
Row 1035: CCCCC[NH+](C)CCC(O)([P](=O)(=O)O)[P](=O)(=O)O
Row 1083: [NH3+]CCC[NH2+]CCS[P](=O)(=O)O

Dropped 15 bad rows. New size: 1314


In [ ]:
# Extract Morgan fingerprint column names that are remaining in the filtered set.

morgan_columns = pd.read_csv(f'{data_dir}/DILI_rank_2_features.csv').columns
morgan_columns = morgan_columns[7:7+2048]
morgan_columns = [i for i in morgan_columns if i in X_filtered.columns]

In [ ]:
# Split like the paper

metadata_cols = ['Label', 'smiles_r', 'InChIKey', 'InChIKey14', 'protonated_smiles_r']
final_base = features[metadata_cols].copy().reset_index(drop=True)

final_features = pd.concat([
    final_base,
    X_filtered.reset_index(drop=True),
    proxy_df.reset_index(drop=True)
], axis=1)

feature_cols = [col for col in final_features.columns 
                if col not in ['Label', 'smiles_r', 'InChIKey', 'InChIKey14', 'protonated_smiles_r']]


smiles     = final_features['smiles_r'].values
activities = final_features['Label'].values

dataset = dc.data.DiskDataset.from_numpy(
    X=activities, 
    ids=smiles
)

splitter = ButinaSplitter()
train_dataset, test_dataset = splitter.train_test_split(
    dataset, 
    cutoff=0.70, 
    seed=42
)

train_smiles = train_dataset.ids
test_smiles  = test_dataset.ids

# Map SMILES back to original row indices in final_features
smiles_to_idx = dict(zip(final_features['smiles_r'], final_features.index))

train_idx = [smiles_to_idx[s] for s in train_smiles]
test_idx  = [smiles_to_idx[s] for s in test_smiles]

# === Slice the FULL feature matrix (with proxy predictions) ===
X_train = final_features.iloc[train_idx][feature_cols].reset_index(drop=True)
X_test  = final_features.iloc[test_idx][feature_cols].reset_index(drop=True)
y_train = final_features.iloc[train_idx]['Label'].reset_index(drop=True)
y_test  = final_features.iloc[test_idx]['Label'].reset_index(drop=True)

print(f"Train size: {len(X_train)}")
print(f"Test size : {len(X_test)}")
print(f"Train toxic ratio: {y_train.mean():.3f} ({y_train.sum()}/{len(y_train)})")
print(f"Test  toxic ratio: {y_test.mean():.3f} ({y_test.sum()}/{len(y_test)})")

Train size: 1051
Test size : 263
Train toxic ratio: 0.538 (565/1051)
Test  toxic ratio: 0.894 (235/263)


In [ ]:
# Force balance toxic in test set
target_ratio = 0.60
current_ratio = y_test.mean()

print(f"Before rebalancing: Test toxic ratio = {current_ratio:.3f}")

while abs(current_ratio - target_ratio) > 0.03 and len(test_idx) > 30:
    if current_ratio > target_ratio:
        
        toxic_test = [i for i in test_idx if final_features.iloc[i]['Label'] == 1]
        if toxic_test:
            move = toxic_test[0]
            test_idx.remove(move)
            train_idx.append(move)
    else:

        no_train = [i for i in train_idx if final_features.iloc[i]['Label'] == 0]
        if no_train:
            move = no_train[0]
            train_idx.remove(move)
            test_idx.append(move)
    
    train_idx = sorted(train_idx)
    test_idx = sorted(test_idx)
    current_ratio = final_features.iloc[test_idx]['Label'].mean()

# Re-slice with the rebalanced indices
X_train = final_features.iloc[train_idx][feature_cols].reset_index(drop=True)
X_test  = final_features.iloc[test_idx][feature_cols].reset_index(drop=True)
y_train = final_features.iloc[train_idx]['Label'].reset_index(drop=True)
y_test  = final_features.iloc[test_idx]['Label'].reset_index(drop=True)

print(f"After rebalancing:  Test toxic ratio = {y_test.mean():.3f} ({y_test.sum()}/{len(y_test)})")
print(f"Train toxic ratio = {y_train.mean():.3f} ({y_train.sum()}/{len(y_train)})")

Before rebalancing: Test toxic ratio = 0.894
After rebalancing:  Test toxic ratio = 0.728 (75/103)
Train toxic ratio = 0.599 (725/1211)


In [102]:
# Train the final 2-class model (nested CV + best threshold)

outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

param_dist_grid = { 
                'max_depth': stats.randint(5, 20),
                'max_features': stats.randint(30, 50),
                'min_samples_leaf': stats.randint(5, 15),
                'min_samples_split': stats.randint(5, 15),
                'n_estimators':[200, 300, 400, 500, 600],
                'bootstrap': [True, False],
                'oob_score': [False],
                'random_state': [42],
                'criterion': ['gini', 'entropy'],
                'n_jobs': [40],
                'class_weight' : [None, 'balanced_subsample', 'balanced']}

rf = RandomForestClassifier(n_jobs=-1, random_state=42)

inner_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

rsh = HalvingRandomSearchCV(
    estimator=rf,
    param_distributions=param_dist_grid,
    factor=2,
    random_state=42,
    n_jobs=-1,
    verbose=1,
    cv=inner_cv
)

print("Running HalvingRandomSearchCV...")
rsh.fit(X_train, y_train)
classifier = rsh.best_estimator_
classifier.fit(X_train, y_train)

print(f"Best parameters: {rsh.best_params_}")

# === 2. Find best threshold using J-statistic on nested CV predictions ===
print("\nFinding best threshold on inner CV...")
cross_val_prob = cross_val_predict(
    classifier, X_train, y_train, 
    cv=inner_cv, method='predict_proba', n_jobs=-1
)[:, 1]

fpr, tpr, thresholds = roc_curve(y_train, cross_val_prob)
J = tpr - fpr
ix = np.argmax(J)
best_thresh = thresholds[ix]
print(f"Best Threshold = {best_thresh:.4f}")

# === 3. Final evaluation on held-out test set ===
y_proba_test = classifier.predict_proba(X_test)[:, 1]
y_pred_test  = (y_proba_test >= best_thresh).astype(int)

cm = confusion_matrix(y_test, y_pred_test)
print("\n=== Confusion Matrix ===")
print(cm)

print("\n=== Performance Metrics ===")
print(f"Balanced Accuracy : {balanced_accuracy_score(y_test, y_pred_test):.4f}")
print(f"Macro F1          : {f1_score(y_test, y_pred_test, average='macro'):.4f}")
print(f"Matthews Corr     : {matthews_corrcoef(y_test, y_pred_test):.4f}")
print(f"Precision (PPV)   : {precision_score(y_test, y_pred_test):.4f}")
print(f"Average Precision : {average_precision_score(y_test, y_proba_test):.4f}")
print(f"ROC-AUC           : {roc_auc_score(y_test, y_proba_test):.4f}")

# Sensitivity / Specificity / LR+
tn, fp, fn, tp = cm.ravel()
sensitivity = tp / (tp + fn)
specificity = tn / (tn + fp)
lr_plus = sensitivity / (1 - specificity) if (1 - specificity) > 0 else np.inf
print(f"Sensitivity       : {sensitivity:.4f}")
print(f"Specificity       : {specificity:.4f}")
print(f"LR+               : {lr_plus:.4f}")

Running HalvingRandomSearchCV...
n_iterations: 6
n_required_iterations: 6
n_possible_iterations: 6
min_resources_: 20
max_resources_: 1211
aggressive_elimination: False
factor: 2
----------
iter: 0
n_candidates: 60
n_resources: 20
Fitting 5 folds for each of 60 candidates, totalling 300 fits
----------
iter: 1
n_candidates: 30
n_resources: 40
Fitting 5 folds for each of 30 candidates, totalling 150 fits
----------
iter: 2
n_candidates: 15
n_resources: 80
Fitting 5 folds for each of 15 candidates, totalling 75 fits
----------
iter: 3
n_candidates: 8
n_resources: 160
Fitting 5 folds for each of 8 candidates, totalling 40 fits
----------
iter: 4
n_candidates: 4
n_resources: 320
Fitting 5 folds for each of 4 candidates, totalling 20 fits
----------
iter: 5
n_candidates: 2
n_resources: 640
Fitting 5 folds for each of 2 candidates, totalling 10 fits
Best parameters: {'bootstrap': False, 'class_weight': None, 'criterion': 'entropy', 'max_depth': 11, 'max_features': 34, 'min_samples_leaf': 7, 

In [112]:
# Check for overfitting
 
train_proba = classifier.predict_proba(X_train)[:, 1]
train_pred  = (train_proba >= best_thresh).astype(int)

# 2. Predictions on test set (you already have these)
test_proba = classifier.predict_proba(X_test)[:, 1]
test_pred  = (test_proba >= best_thresh).astype(int)

print("=== OVERFITTING CHECK ===")
print(f"Train Balanced Accuracy : {balanced_accuracy_score(y_train, train_pred):.4f}")
print(f"Test  Balanced Accuracy : {balanced_accuracy_score(y_test,  test_pred):.4f}")
print(f"Train vs Test gap       : {balanced_accuracy_score(y_train, train_pred) - balanced_accuracy_score(y_test, test_pred):.4f}")

print(f"\nTrain Sensitivity       : {recall_score(y_train, train_pred):.4f}")
print(f"Test  Sensitivity       : {recall_score(y_test,  test_pred):.4f}")

print(f"\nInner CV best score     : {rsh.best_score_:.4f}")
print(f"Test Balanced Accuracy  : {balanced_accuracy_score(y_test, test_pred):.4f}")

=== OVERFITTING CHECK ===
Train Balanced Accuracy : 0.9717
Test  Balanced Accuracy : 0.6593
Train vs Test gap       : 0.3124

Train Sensitivity       : 0.9434
Test  Sensitivity       : 0.6400

Inner CV best score     : 0.6666
Test Balanced Accuracy  : 0.6593


In [108]:
# Try different thresholds

for thresh in [0.75, 0.7, 0.65, 0.6, 0.55, 0.5, 0.45, 0.40, 0.35, 0.30]:
    y_pred = (classifier.predict_proba(X_test)[:,1] >= thresh).astype(int)
    
    # Calculate metrics
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    
    sensitivity = recall_score(y_test, y_pred)          # TP / (TP + FN)
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    lr_plus = sensitivity / (1 - specificity) if (1 - specificity) > 0 else np.inf
    
    print(f"Threshold = {thresh:.2f} | "
          f"Sensitivity = {sensitivity:.3f} | "
          f"Specificity = {specificity:.3f} | "
          f"LR+ = {lr_plus:.2f}")

Threshold = 0.75 | Sensitivity = 0.133 | Specificity = 1.000 | LR+ = inf
Threshold = 0.70 | Sensitivity = 0.427 | Specificity = 0.964 | LR+ = 11.95
Threshold = 0.65 | Sensitivity = 0.640 | Specificity = 0.679 | LR+ = 1.99
Threshold = 0.60 | Sensitivity = 0.787 | Specificity = 0.464 | LR+ = 1.47
Threshold = 0.55 | Sensitivity = 0.867 | Specificity = 0.286 | LR+ = 1.21
Threshold = 0.50 | Sensitivity = 0.920 | Specificity = 0.179 | LR+ = 1.12
Threshold = 0.45 | Sensitivity = 0.933 | Specificity = 0.143 | LR+ = 1.09
Threshold = 0.40 | Sensitivity = 0.960 | Specificity = 0.036 | LR+ = 1.00
Threshold = 0.35 | Sensitivity = 0.987 | Specificity = 0.000 | LR+ = 0.99
Threshold = 0.30 | Sensitivity = 1.000 | Specificity = 0.000 | LR+ = 1.00


In [109]:
# Results for best threshold

best_thresh = 0.65
y_pred_test = (classifier.predict_proba(X_test)[:,1] >= best_thresh).astype(int)

cm = confusion_matrix(y_test, y_pred_test)
print("Confusion Matrix:\n", cm)

sens = recall_score(y_test, y_pred_test)
spec = cm[0,0] / (cm[0,0] + cm[0,1]) if (cm[0,0] + cm[0,1]) > 0 else 0
lr_plus = sens / (1 - spec) if (1 - spec) > 0 else np.inf

print(f"Sensitivity : {sens:.3f}")
print(f"LR+         : {lr_plus:.2f}")
print(f"Balanced Acc: {balanced_accuracy_score(y_test, y_pred_test):.3f}")

print(f"Test positives: {y_test.sum()} / {len(y_test)}")

Confusion Matrix:
 [[19  9]
 [27 48]]
Sensitivity : 0.640
LR+         : 1.99
Balanced Acc: 0.659
Test positives: 75 / 103


In [110]:
# TOP 25
probs = classifier.predict_proba(X_test)[:, 1]
top_idx = np.argsort(probs)[::-1][:25]                  # indices of top 25

top_true = y_test.iloc[top_idx].values
top_pred = np.ones(25)

tp = np.sum(top_true == 1)
fp = np.sum(top_true == 0)
precision = tp / 25

sensitivity_top25 = tp / y_test.sum()
specificity_top25 = 1 - (fp / (len(y_test) - y_test.sum()))
lr_plus_top25 = sensitivity_top25 / (1 - specificity_top25) if (1 - specificity_top25) > 0 else np.inf

print("=== TOP-25 RESULTS ===")
print(f"Number of actual toxic in top 25 : {tp} out of 25")
print(f"Precision (PPV) in top 25       : {precision:.3f}")
print(f"Sensitivity (recall) in top 25  : {sensitivity_top25:.3f}")
print(f"LR+ for top 25                   : {lr_plus_top25:.2f}")

=== TOP-25 RESULTS ===
Number of actual toxic in top 25 : 24 out of 25
Precision (PPV) in top 25       : 0.960
Sensitivity (recall) in top 25  : 0.320
LR+ for top 25                   : 8.96
